In [2]:
# imports
import spacy
import json

ModuleNotFoundError: No module named 'spacy'

In [ ]:
# variable declaration
nlp = spacy.load("de_core_news_lg")

text = "Ein Kunde kann mehrere Bestellungen aufgeben. Jede Bestellung enthält mehrere Produkte. Jeder Kunde hat einen Namen."
doc = nlp(text)

entities = {}
relations = []

entity_id_counter = 1


In [ ]:
# function declarations

### Entity and relation extraction functions
def get_quantity(noun):
    for child in noun.children:
        if child.pos_ in ("DET", "NUM"):
            return normalize_quantity(child, noun)
    return "m"  # default

### Entity management functions
def get_entity_id(noun):
    global entity_id_counter
    key = noun.lemma_
    if key not in entities:
        entities[key] = {
            "id": f"e{entity_id_counter}",
            "type": noun.lemma_,
            #"quantity": get_quantity(noun)
        }
        entity_id_counter += 1
    return entities[key]["id"]

### Quantity normalization function
def normalize_quantity(token, noun):
    quantity_map = {
        "ein": "1",
        "eine": "1",
        "jede": "1",
        "jeder": "1",
        "jedes": "1",
        "mehrere": "n",
        "viele": "n",
        "alle": "n",
        "einige": "n",
        "kein": "0",
        "keine": "0",
    }

    if token.pos_ == "NUM":
        return token.text
    elif noun.morph.get("Number") == ["Plur"]:
        return "n"
    elif token.pos_ == "DET":
        text_lower = token.text.lower()
        return quantity_map.get(text_lower, "m")
    return "m"

# Verb extraction function
def get_main_verb(sent):
    root = sent.root

    # Falls ROOT ein Modal- oder Hilfsverb ist
    if root.pos_ in ["AUX", "VERB"]:
        for child in root.children:
            if child.dep_ in ["xcomp", "oc"] and child.pos_ == "VERB":
                return child.lemma_

    return root.lemma_


In [ ]:
token = nlp("kann")[0]
print(token.lemma_)

In [ ]:
# Main - extraction loop

# Extract entities and relations
for sent in doc.sents:  # Loop through sentences
    
    # Extract subject, verb, and object
    subject = None
    obj = None
    verb = get_main_verb(sent)

    # Identify subject and object based on dependency labels
    for token in sent:
        if token.dep_ in ("sb", "nsubj"):
            subject = token
        elif token.dep_ in ("oa", "obj"):
            obj = token

    # Only create a relation if we have a valid subject, verb, and object
    if subject and verb and obj:
        subj_id = get_entity_id(subject)
        obj_id = get_entity_id(obj)

        # Add relation with cardinality information
        relations.append({
            "subject": subj_id,
            "predicate": verb,
            "object": obj_id,
            "cardinality": {
                "subject": get_quantity(subject),
                "object": get_quantity(obj)
            }
        })

output = {
    "entities": list(entities.values()),
    "relations": relations
}

print(json.dumps(output, indent=2, ensure_ascii=False))


aufgeben
enthalten
haben
{
  "entities": [
    {
      "id": "e1",
      "type": "Kunde"
    },
    {
      "id": "e2",
      "type": "Bestellung"
    },
    {
      "id": "e3",
      "type": "Produkt"
    },
    {
      "id": "e4",
      "type": "Name"
    }
  ],
  "relations": [
    {
      "subject": "e1",
      "predicate": "aufgeben",
      "object": "e2",
      "cardinality": {
        "subject": "1",
        "object": "n"
      }
    },
    {
      "subject": "e2",
      "predicate": "enthalten",
      "object": "e3",
      "cardinality": {
        "subject": "1",
        "object": "n"
      }
    },
    {
      "subject": "e1",
      "predicate": "haben",
      "object": "e4",
      "cardinality": {
        "subject": "1",
        "object": "m"
      }
    },
    {
      "subject": "e1",
      "predicate": "aufgeben",
      "object": "e2",
      "cardinality": {
        "subject": "1",
        "object": "n"
      }
    },
    {
      "subject": "e2",
      "predicate": "enthalt

In [ ]:
for sent in doc.sents:  # Loop through sentences
    
    # Extract subject, verb, and object
    subject = None
    obj = None
    verb = get_main_verb(sent)

    # Identify subject and object based on dependency labels
    for token in sent:
        if token.dep_ in ("sb", "nsubj"):
            subject = token
        elif token.dep_ in ("oa", "obj"):
            obj = token

    # Only create a relation if we have a valid subject, verb, and object
    if subject and verb and obj:
        subj_id = get_entity_id(subject)
        obj_id = get_entity_id(obj)

        # Add relation with cardinality information
        relations.append({
            "subject": subj_id,
            "predicate": verb,
            "object": obj_id,
            "cardinality": {
                "subject": get_quantity(subject),
                "object": get_quantity(obj)
            }
        })

output = {
    "entities": list(entities.values()),
    "relations": relations
}

print(json.dumps(output, indent=2, ensure_ascii=False))


In [ ]:
for token in doc:
    print(token.text, token.dep_, token.head.text)

Ein nk Kunde
Kunde sb kann
kann ROOT kann
mehrere nk Bestellungen
Bestellungen oa aufgeben
aufgeben oc kann
. punct kann
Jede nk Bestellung
Bestellung sb enthält
enthält ROOT enthält
mehrere nk Produkte
Produkte oa enthält
. punct enthält
Jeder nk Kunde
Kunde sb hat
hat ROOT hat
einen nk Namen
Namen oa hat
. punct hat
